# Lateral Double Quantum Dot with Top Barriers

This notebook rebuilds the lateral double dot from
`03_generate_nextnano_input_from_phidl_layout.ipynb`. The only intended layout
difference is that `B1`, `B2`, and `B3` approach from the **top**, the same side
as `P1` and `P2`.

Plunger geometry is not reimplemented in the notebook. Both the reference
bottom-barrier array and this top-barrier array use the same shared
`PlungerGate` and `CANONICAL_LOLLIPOP_PLUNGER` geometry contract. The notebook
checks that their plunger polygons are point-for-point identical before writing
the nextnano input.

The workflow includes:

- an exact top-down polygon view, where the rectangular stem and faceted head
  are unambiguous
- a polygon-preserving 3D view from the shared `qd_design` renderer
- an optional structure-only nextnano run with every solver disabled
- a notebook-03-style view of the actual nextnano `Structure` output
- `integrated_hole_density`, plane cuts, and line cuts

## 0. Setup

In [23]:
import sys
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

REPO_ROOT = next(
    path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (path / "src").exists() and (path / "notebooks").exists()
)
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from qd_design import (
    CANONICAL_LOLLIPOP_PLUNGER,
    LinearDotArrayDevice,
    TopBarrierLinearDotArrayDevice,
    build_simulation_layout,
    make_reference_sige_ge_process_stack,
    plot_layout_spec_2d,
    plot_simulation_layout_3d,
    write_nextnano_input_from_template,
)
import nextnanopp_tools as nnt


def print_json(value):
    print(json.dumps(value, indent=2))


print("REPO_ROOT:", REPO_ROOT)

REPO_ROOT: /Users/robertjovanov/code/qpu-design-automation-toolkit


In [24]:
OUTPUT_DIR = REPO_ROOT / "data" / "gds"
GENERATED_INPUT_DIR = REPO_ROOT / "configs" / "robert_inputs" / "generated"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
GENERATED_INPUT_DIR.mkdir(parents=True, exist_ok=True)

TEMPLATE_INPUT_PATH = (
    REPO_ROOT / "configs" / "robert_inputs" / "double_qd" / "3d"
    / "Double_Quantum_Dot_3D.in"
)
GDS_PATH = OUTPUT_DIR / "lateral_double_dot_top_barriers.gds"
SVG_PATH = OUTPUT_DIR / "lateral_double_dot_top_barriers.svg"
LAYOUT_SPEC_PATH = OUTPUT_DIR / "lateral_double_dot_top_barriers_layout_spec.json"
SIMULATION_LAYOUT_PATH = OUTPUT_DIR / "lateral_double_dot_top_barriers_simulation_layout.json"
GENERATED_INPUT_PATH = GENERATED_INPUT_DIR / "Lateral_Double_Quantum_Dot_Top_Barriers.in"

RUNS_DIR = REPO_ROOT / "runs"
RUN_TAG = "structure_only_top_barriers"
RUN_SIMULATION = False  # Set True to launch nextnano++.
BIAS_INDEX = 0
RUN_VARIABLE_OVERRIDES = {
    "strain": 0,
    "poisson": 0,
    "quantum": 0,
    "quantum_poisson": 0,
}

print("Template exists:", TEMPLATE_INPUT_PATH.exists())
print("RUN_SIMULATION:", RUN_SIMULATION)
print("Structure-only switches:", RUN_VARIABLE_OVERRIDES)

Template exists: True
RUN_SIMULATION: False
Structure-only switches: {'strain': 0, 'poisson': 0, 'quantum': 0, 'quantum_poisson': 0}


## 1. Build the reference and top-barrier arrays from the same primitive

In [25]:
# These are the shared defaults used by the array builders in notebook 03,
# single-dot, vertical-stack, square-array, and this top-barrier design.
print_json(CANONICAL_LOLLIPOP_PLUNGER.to_dict())

common_array_parameters = dict(
    n_dots=2,
    device_y_size_nm=200.0,
    ohmic_width_nm=40.0,
    ohmic_length_nm=200.0,
    barrier_width_nm=40.0,
    barrier_length_nm=140.0,
    ohmic_to_barrier_gap_nm=20.0,
    barrier_to_plunger_gap_nm=20.0,
)

# Reference geometry from notebook 03: barriers from below, plungers from above.
reference_double_dot = LinearDotArrayDevice(
    name="reference_bottom_barrier_double_dot",
    **common_array_parameters,
)

# New geometry: the same canonical plungers, with barriers moved to the top.
double_dot = TopBarrierLinearDotArrayDevice(
    name="lateral_double_dot_top_barriers",
    **common_array_parameters,
)

reference_double_dot.ensure_built()
layout = double_dot.ensure_built()
layout_spec = double_dot.layout_spec()

print_json(double_dot.summary())

{
  "body_width_nm": 40.0,
  "body_length_nm": 50.0,
  "head_top_width_nm": 60.0,
  "head_max_width_nm": 100.0,
  "head_height_nm": 100.0,
  "upper_taper_height_nm": 25.0,
  "lower_taper_height_nm": 25.0
}
{
  "name": "lateral_double_dot_top_barriers",
  "n_dots": 2,
  "device_y_size_nm": 200.0,
  "ohmic_width_nm": 40.0,
  "ohmic_length_nm": 200.0,
  "barrier_width_nm": 40.0,
  "barrier_length_nm": 140.0,
  "barrier_side": "top",
  "plunger_body_width_nm": 40.0,
  "plunger_body_length_nm": 50.0,
  "plunger_head_top_width_nm": 60.0,
  "plunger_head_max_width_nm": 100.0,
  "plunger_head_height_nm": 100.0,
  "plunger_upper_taper_height_nm": 25.0,
  "plunger_lower_taper_height_nm": 25.0,
  "ohmic_to_barrier_gap_nm": 20.0,
  "barrier_to_plunger_gap_nm": 20.0,
  "include_screening_gates": false,
  "screening_gate_width_nm": 40.0,
  "screening_gate_length_nm": 50.0
}


### 1.1 Exact top-down layout: lollipop outlines

In [26]:
# This view draws polygon vertices directly; no bounding boxes are involved.
layout_plan_fig = plot_layout_spec_2d(
    layout_spec,
    title="Top-barrier double dot: exact PHIDL polygon outlines",
)
layout_plan_fig.show()

### 1.2 Point-for-point comparison with notebook 03

In [27]:
def elements_by_name(builder):
    return {element["name"]: element for element in builder.layout_spec()}


reference_elements = elements_by_name(reference_double_dot)
top_barrier_elements = elements_by_name(double_dot)

comparison_rows = []
for name in ("P1", "P2"):
    reference_polygons = reference_elements[name]["polygon_xy_nm"]
    top_barrier_polygons = top_barrier_elements[name]["polygon_xy_nm"]
    identical = reference_polygons == top_barrier_polygons
    comparison_rows.append(
        {
            "gate": name,
            "reference_vertex_counts": [len(p) for p in reference_polygons],
            "top_barrier_vertex_counts": [len(p) for p in top_barrier_polygons],
            "point_for_point_identical": identical,
        }
    )
    assert identical
    assert [len(polygon) for polygon in top_barrier_polygons] == [10]

comparison = pd.DataFrame(comparison_rows)
display(comparison)

barrier_rows = []
for name in ("B1", "B2", "B3"):
    reference = reference_elements[name]
    top = top_barrier_elements[name]
    barrier_rows.append(
        {
            "gate": name,
            "reference_y_nm": (reference["y_min_nm"], reference["y_max_nm"]),
            "top_barrier_y_nm": (top["y_min_nm"], top["y_max_nm"]),
        }
    )
display(pd.DataFrame(barrier_rows))

print("P1/P2 are exactly the same lollipop polygons as notebook 03.")
print("Only the barrier-side placement changes.")

,gate,reference_vertex_counts,top_barrier_vertex_counts,point_for_point_identical
0,P1,[10],[10],True
1,P2,[10],[10],True


,gate,reference_y_nm,top_barrier_y_nm
0,B1,"(0.0, 140.0)","(60.0, 200.0)"
1,B2,"(0.0, 140.0)","(60.0, 200.0)"
2,B3,"(0.0, 140.0)","(60.0, 200.0)"


P1/P2 are exactly the same lollipop polygons as notebook 03.
Only the barrier-side placement changes.


In [28]:
double_dot.write_gds(str(GDS_PATH))
double_dot.write_svg(str(SVG_PATH))
double_dot.write_layout_spec_json(str(LAYOUT_SPEC_PATH))

print("GDS:", GDS_PATH)
print("SVG:", SVG_PATH)
print("Layout spec:", LAYOUT_SPEC_PATH)

GDS: /Users/robertjovanov/code/qpu-design-automation-toolkit/data/gds/lateral_double_dot_top_barriers.gds
SVG: /Users/robertjovanov/code/qpu-design-automation-toolkit/data/gds/lateral_double_dot_top_barriers.svg
Layout spec: /Users/robertjovanov/code/qpu-design-automation-toolkit/data/gds/lateral_double_dot_top_barriers_layout_spec.json


## 2. Build and inspect the 3D simulation layout

In [29]:
process_stack = make_reference_sige_ge_process_stack()
simulation_layout = build_simulation_layout(
    name="lateral_double_dot_top_barriers_3d",
    layout_elements=layout_spec,
    process_stack=process_stack,
    x_margin_nm=0.0,
    y_margin_nm=0.0,
)
simulation_layout.write_json(str(SIMULATION_LAYOUT_PATH))

patterned_summary = pd.DataFrame(
    {
        "name": region.name,
        "gate_type": region.gate_type,
        "polygon_count": len(region.polygon_xy_nm),
        "vertex_counts": [len(polygon) for polygon in region.polygon_xy_nm],
        "z_min_nm": region.z_min_nm,
        "z_max_nm": region.z_max_nm,
    }
    for region in simulation_layout.patterned_regions
)
display(patterned_summary)

,name,gate_type,polygon_count,vertex_counts,z_min_nm,z_max_nm
0,OC_L,ohmic,1,[4],-77.0,173.0
1,B1,barrier,1,[4],108.0,138.0
2,P1,plunger,1,[10],143.0,173.0
3,B2,barrier,1,[4],108.0,138.0
4,P2,plunger,1,[10],143.0,173.0
5,B3,barrier,1,[4],108.0,138.0
6,OC_R,ohmic,1,[4],-77.0,173.0


### 2.1 Polygon-preserving 3D structure

This uses the shared `qd_design.plot_simulation_layout_3d` renderer. Patterned
regions are triangulated from their true polygon vertices and outlined in
black. The camera is intentionally elevated so the lollipop plan shape remains
visible. Set `STRUCTURE_Z_RANGE_NM = None` for the full substrate depth.

In [30]:
STRUCTURE_Z_RANGE_NM = (-120.0, 180.0)

structure_preview_fig = plot_simulation_layout_3d(
    simulation_layout,
    z_range_nm=STRUCTURE_Z_RANGE_NM,
    show_polygon_outlines=True,
)
structure_preview_fig.show()

## 3. Write and validate the structure-only nextnano input

In [31]:
if not TEMPLATE_INPUT_PATH.exists():
    raise FileNotFoundError(TEMPLATE_INPUT_PATH)

write_nextnano_input_from_template(
    simulation_layout=simulation_layout,
    template_path=TEMPLATE_INPUT_PATH,
    output_path=GENERATED_INPUT_PATH,
    voltage_overrides={
        "V_P1": -3.0,
        "V_P2": -3.0,
        "V_B1": 0.0,
        "V_B2": 0.0,
        "V_B3": 0.0,
        "V_OC_L": 0.0,
        "V_OC_R": 0.0,
    },
)

generated_input = nnt.load_input_file(GENERATED_INPUT_PATH)
nnt.set_input_variables(generated_input, RUN_VARIABLE_OVERRIDES)
nnt.save_input_file(generated_input, fullpath=GENERATED_INPUT_PATH, overwrite=True)

generated_text = GENERATED_INPUT_PATH.read_text(encoding="utf-8")
solver_values = {
    name: (
        match.group(1).strip()
        if (match := re.search(rf"^\${name}\s*=\s*([^#\n]+)", generated_text, re.MULTILINE))
        else None
    )
    for name in RUN_VARIABLE_OVERRIDES
}
assert all(str(value) == "0" for value in solver_values.values())

print("Generated input:", GENERATED_INPUT_PATH)
print("Solver switches:", solver_values)

Generated input: /Users/robertjovanov/code/qpu-design-automation-toolkit/configs/robert_inputs/generated/Lateral_Double_Quantum_Dot_Top_Barriers.in
Solver switches: {'strain': '0', 'poisson': '0', 'quantum': '0', 'quantum_poisson': '0'}


In [32]:
def patterned_region_text(input_text, region_name):
    marker = f"# patterned region: {region_name}"
    start = input_text.index(marker)
    next_marker = input_text.find("# patterned region:", start + len(marker))
    return input_text[start:] if next_marker < 0 else input_text[start:next_marker]


export_checks = []
for region in simulation_layout.patterned_regions:
    block = patterned_region_text(generated_text, region.name)
    expected_vertices = sum(len(polygon) for polygon in region.polygon_xy_nm)
    written_vertices = block.count("vertex{")
    export_checks.append(
        {
            "name": region.name,
            "gate_type": region.gate_type,
            "expected_vertices": expected_vertices,
            "written_vertices": written_vertices,
            "uses_polygonal_prism": "polygonal_prism{" in block,
        }
    )

export_checks = pd.DataFrame(export_checks)
display(export_checks)
assert (export_checks["expected_vertices"] == export_checks["written_vertices"]).all()
assert export_checks.loc[
    export_checks["gate_type"] == "plunger", "written_vertices"
].tolist() == [10, 10]

print("Verified: nextnano receives both plungers as the same 10-vertex lollipop prisms.")

,name,gate_type,expected_vertices,written_vertices,uses_polygonal_prism
0,OC_L,ohmic,4,4,True
1,B1,barrier,4,4,True
2,P1,plunger,10,10,True
3,B2,barrier,4,4,True
4,P2,plunger,10,10,True
5,B3,barrier,4,4,True
6,OC_R,ohmic,4,4,True


Verified: nextnano receives both plungers as the same 10-vertex lollipop prisms.


## 4. Optionally run nextnano in structure-only mode

In [33]:
if RUN_SIMULATION:
    RUNS_DIR.mkdir(parents=True, exist_ok=True)
    input_file = nnt.run_input_file(
        GENERATED_INPUT_PATH,
        output_root=RUNS_DIR,
        tag=RUN_TAG,
        add_timestamp=True,
        variables=RUN_VARIABLE_OVERRIDES,
        show_log=True,
        convergenceCheck=False,
        staging_root=GENERATED_INPUT_DIR / "staged_runs",
        keep_staged_input=True,
    )
    OUTPUT_RUN_DIR = nnt.get_output_directory(input_file)
    print("Structure-only run completed:", OUTPUT_RUN_DIR)
else:
    OUTPUT_RUN_DIR = None
    print("RUN_SIMULATION is False. Set it to True and rerun this cell to launch nextnano.")

RUN_SIMULATION is False. Set it to True and rerun this cell to launch nextnano.


In [34]:
RUNS_DIR.mkdir(parents=True, exist_ok=True)
if OUTPUT_RUN_DIR is None:
    matching_runs = nnt.find_runs_for_input(RUNS_DIR, GENERATED_INPUT_PATH)
    tagged_runs = [run for run in matching_runs if RUN_TAG in run.name]
    if tagged_runs:
        matching_runs = tagged_runs
    OUTPUT_RUN_DIR = matching_runs[-1] if matching_runs else None

RUN_AVAILABLE = OUTPUT_RUN_DIR is not None and Path(OUTPUT_RUN_DIR).exists()
if RUN_AVAILABLE:
    OUTPUT_RUN_DIR = nnt.resolve_run_root(OUTPUT_RUN_DIR)
    STRUCTURE_DIR = OUTPUT_RUN_DIR / "Structure"
    print("Selected run:", OUTPUT_RUN_DIR)
    print("Structure directory:", STRUCTURE_DIR)
else:
    STRUCTURE_DIR = None
    print("No matching completed run is available yet.")

Selected run: /Users/robertjovanov/code/qpu-design-automation-toolkit/runs/Lateral_Double_Quantum_Dot_Top_Barriers
Structure directory: /Users/robertjovanov/code/qpu-design-automation-toolkit/runs/Lateral_Double_Quantum_Dot_Top_Barriers/Structure


## 5. Plot the actual nextnano `Structure` output

Like notebook 03, this section reads `materials.vtr` and `contacts.vtr` from
the run's `Structure` directory. Gate points therefore come from nextnano's
contact-index grid, providing an independent post-export geometry check.

In [35]:
VTR_DATA_ARRAY_RE = re.compile(
    r'<DataArray[^>]*Name\s*=\s*"([^"]+)"[^>]*>(.*?)</DataArray>',
    flags=re.DOTALL,
)
COORDINATE_ARRAY_NAMES = ("X_COORDINATES", "Y_COORDINATES", "Z_COORDINATES")


def read_index_lookup(path):
    path = Path(path)
    if not path.exists():
        return pd.DataFrame(columns=["Index"])
    lookup = pd.read_csv(path, sep=r"\s+")
    lookup["Index"] = lookup["Index"].astype(int)
    return lookup


def read_ascii_vtr_index(path, dtype=np.int32):
    path = Path(path)
    text = path.read_text(encoding="utf-8")
    arrays = {}
    for name, body in VTR_DATA_ARRAY_RE.findall(text):
        name = name.strip()
        array_dtype = np.float64 if name in COORDINATE_ARRAY_NAMES else np.float32
        arrays[name] = np.fromstring(body, sep=" ", dtype=array_dtype)

    missing = [name for name in COORDINATE_ARRAY_NAMES if name not in arrays]
    if missing:
        raise ValueError(f"Missing coordinate arrays in {path.name}: {missing}")
    data_names = [name for name in arrays if name not in COORDINATE_ARRAY_NAMES]
    if len(data_names) != 1:
        raise ValueError(f"Expected one data array in {path.name}, got {data_names}")

    x, y, z = (arrays[name] for name in COORDINATE_ARRAY_NAMES)
    values = np.rint(arrays[data_names[0]]).astype(dtype, copy=False)
    expected = len(x) * len(y) * len(z)
    if values.size != expected:
        raise ValueError(f"{path.name}: expected {expected:,} values, got {values.size:,}")
    return {
        "x": x,
        "y": y,
        "z": z,
        "values": values.reshape((len(z), len(y), len(x))),
    }


def sample_contact_points(contact_grid, contact_index, z_range_nm, max_points=3500):
    mask = contact_grid["values"] == int(contact_index)
    if z_range_nm is not None:
        z_min, z_max = z_range_nm
        z_mask = (contact_grid["z"] >= z_min) & (contact_grid["z"] <= z_max)
        mask = mask & z_mask[:, None, None]
    zz, yy, xx = np.nonzero(mask)
    if xx.size == 0:
        return None
    if xx.size > max_points:
        selection = np.linspace(0, xx.size - 1, max_points, dtype=int)
        zz, yy, xx = zz[selection], yy[selection], xx[selection]
    return contact_grid["x"][xx], contact_grid["y"][yy], contact_grid["z"][zz]

In [36]:
if RUN_AVAILABLE and STRUCTURE_DIR.exists():
    contact_lookup = read_index_lookup(STRUCTURE_DIR / "contact_indices.txt")
    contact_grid = read_ascii_vtr_index(STRUCTURE_DIR / "contacts.vtr", dtype=np.int16)

    contact_name_to_index = dict(zip(contact_lookup["Contact"], contact_lookup["Index"]))
    surface_gate_names = ["OC_L", "B1", "P1", "B2", "P2", "B3", "OC_R"]
    colors = ["#984ea3", "#ff7f00", "#e41a1c", "#ff7f00", "#e41a1c", "#ff7f00", "#984ea3"]

    nextnano_structure_fig = go.Figure()
    for name, color in zip(surface_gate_names, colors):
        if name not in contact_name_to_index:
            continue
        sampled = sample_contact_points(
            contact_grid,
            contact_name_to_index[name],
            z_range_nm=(-120.0, 180.0),
        )
        if sampled is None:
            continue
        xs, ys, zs = sampled
        nextnano_structure_fig.add_trace(go.Scatter3d(
            x=xs,
            y=ys,
            z=zs,
            mode="markers",
            name=name,
            marker={"size": 2.8, "color": color, "opacity": 0.86},
            hovertemplate=f"{name}<br>x=%{{x:.1f}} nm<br>y=%{{y:.1f}} nm<br>z=%{{z:.1f}} nm<extra></extra>",
        ))

    nextnano_structure_fig.update_layout(
        title="Actual nextnano contact-index structure (z = -120 to 180 nm)",
        height=760,
        margin={"l": 0, "r": 0, "t": 48, "b": 0},
        scene={
            "xaxis_title": "x (nm)",
            "yaxis_title": "y (nm)",
            "zaxis_title": "z (nm)",
            "aspectmode": "manual",
            "aspectratio": {"x": 1.4, "y": 0.8, "z": 0.8},
            "camera": {
                "eye": {"x": 1.15, "y": -1.35, "z": 2.15},
                "projection": {"type": "orthographic"},
            },
        },
    )
    nextnano_structure_fig.show()
else:
    print("Run section 4 first to plot the actual nextnano Structure output.")

Run section 4 first to plot the actual nextnano Structure output.


## 6. Analyze the completed Poisson + quantum-Poisson run

The cells below use this run directly, independently of the optional local run
in section 4:

`runs/Lateral_Double_Quantum_Dot_Top_Barriers`

In [37]:
ANALYSIS_RUN_DIR = REPO_ROOT / "runs" / "Lateral_Double_Quantum_Dot_Top_Barriers"
ANALYSIS_BIAS_DIR = ANALYSIS_RUN_DIR / "bias_00000"
ANALYSIS_QUANTUM_DIR = ANALYSIS_BIAS_DIR / "Quantum"

if not ANALYSIS_RUN_DIR.exists():
    raise FileNotFoundError(ANALYSIS_RUN_DIR)

JOB_DONE_PATH = ANALYSIS_RUN_DIR / "job_done.txt"
job_done_text = JOB_DONE_PATH.read_text(encoding="utf-8").strip() if JOB_DONE_PATH.exists() else ""
JOB_FINISHED_SUCCESSFULLY = (
    JOB_DONE_PATH.exists()
    and "successfully completed" in job_done_text.lower()
)

job_status = pd.Series(
    {
        "run_directory": str(ANALYSIS_RUN_DIR),
        "job_done_exists": JOB_DONE_PATH.exists(),
        "job_done_message": job_done_text,
        "finished_successfully": JOB_FINISHED_SUCCESSFULLY,
        "bias_directory_exists": ANALYSIS_BIAS_DIR.exists(),
        "quantum_directory_exists": ANALYSIS_QUANTUM_DIR.exists(),
    },
    name="value",
)
display(job_status.to_frame())

if not JOB_FINISHED_SUCCESSFULLY:
    raise RuntimeError(
        f"Run did not report successful completion in {JOB_DONE_PATH}: {job_done_text!r}"
    )

,value
run_directory,/Users/robertjovanov/code/qpu-design-automatio...
job_done_exists,True
job_done_message,Calculation successfully completed.
finished_successfully,True
bias_directory_exists,True
quantum_directory_exists,True


### 6.1 Integrated hole density

In [38]:
INTEGRATED_DENSITY_PATH = ANALYSIS_RUN_DIR / "integrated_density_hole.dat"
integrated_density_hole = nnt.read_integrated_density_hole(INTEGRATED_DENSITY_PATH)
integrated_region_columns = nnt.integrated_density_region_columns(integrated_density_hole)

display(integrated_density_hole)

integrated_density_summary = pd.DataFrame(
    {
        "region": integrated_region_columns,
        "integrated_holes": [
            float(integrated_density_hole[column].iloc[-1])
            for column in integrated_region_columns
        ],
    }
)
integrated_density_summary.loc[len(integrated_density_summary)] = {
    "region": "all reported regions",
    "integrated_holes": integrated_density_summary["integrated_holes"].sum(),
}
display(integrated_density_summary)

,OC_L_bias[V],B1_bias[V],P1_bias[V],B2_bias[V],P2_bias[V],B3_bias[V],OC_R_bias[V],Body_bias[V],remove_surface_charge_bias[V],zero_fermi_QW_bias[V],region_2[carriers]
0,0,0,0,0,0,0,0,0,-1.5,0,20.201432


,region,integrated_holes
0,region_2[carriers],20.201432
1,all reported regions,20.201432


### 6.2 Total-charge summary

In [39]:
TOTAL_CHARGES_PATH = ANALYSIS_BIAS_DIR / "total_charges.txt"
total_charges = nnt.read_total_charges(TOTAL_CHARGES_PATH)
display(total_charges)

charge_values = total_charges.set_index("quantity")["value"]
total_charge_summary = pd.Series(
    {
        "electrons_e": float(charge_values.get("electrons", np.nan)),
        "holes_e": float(charge_values.get("holes", np.nan)),
        "ionized_donors_e": float(charge_values.get("ionized donors", np.nan)),
        "ionized_acceptors_e": float(charge_values.get("ionized acceptors", np.nan)),
        "fixed_charges_e": float(charge_values.get("fixed charges", np.nan)),
        "polarization_total_e": float(charge_values.get("polarization_total", np.nan)),
        "reported_sum_e": float(charge_values.get("sum", np.nan)),
    },
    name="value",
)
display(total_charge_summary.to_frame())

,quantity,value,unit
0,electrons,-0.00000,e
1,holes,42.77874,e
2,ionized donors,0.00000,e
3,ionized acceptors,-0.00000,e
4,fixed charges,0.00000,e
5,polarization_total,0.00000,e
6,polarization_piezo,0.00000,e
7,sum,42.77874,e


,value
electrons_e,-0.00000
holes_e,42.77874
ionized_donors_e,0.00000
ionized_acceptors_e,-0.00000
fixed_charges_e,0.00000
polarization_total_e,0.00000
reported_sum_e,42.77874


## 7. Reusable VTR plane- and line-cut extraction

`extract_vtr_cuts` reads a chosen plane and line directly from an ASCII VTR
file. The plotting helpers intentionally create **separate figures** and use
the raw linear values. For the default `z` plane and `x` line, the readers use
memory-mapped windows and avoid loading the complete 3D volume.

In [40]:
def extract_vtr_cuts(
    path,
    *,
    variable,
    plane_axis="z",
    plane_value_nm=-3.75,
    line_axis="x",
    line_fixed_coords_nm=None,
):
    # Extract one plane and one line from a nextnano ASCII VTR output.
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)

    available_variables = nnt.list_variables(path)
    if variable not in available_variables:
        raise KeyError(
            f"{variable!r} is not present in {path.name}; "
            f"available variables: {available_variables}"
        )

    if line_fixed_coords_nm is None:
        line_fixed_coords_nm = {"y": 100.0, "z": plane_value_nm}

    plane = nnt.load_vtr_plane(
        path,
        variable=variable,
        slice_axis=plane_axis,
        slice_value=plane_value_nm,
    )
    line = nnt.load_vtr_linecut(
        path,
        variable=variable,
        axis=line_axis,
        fixed_coords=line_fixed_coords_nm,
    )
    return {
        "path": path,
        "variable": variable,
        "available_variables": available_variables,
        "plane": plane,
        "line": line,
    }


def plane_values_yx(plane):
    # Return values in the orientation expected by Plotly: (y, x).
    values = np.asarray(plane["values"], dtype=float)
    if values.shape == (len(plane["x"]), len(plane["y"])):
        return values.T
    if values.shape == (len(plane["y"]), len(plane["x"])):
        return values
    raise ValueError(
        f"Cannot orient plane shape {values.shape} against "
        f"x={len(plane['x'])}, y={len(plane['y'])}."
    )


def plot_vtr_plane(
    cuts,
    *,
    title=None,
    colorscale="Turbo",
    color_range=None,
):
    # Plot one raw, linear-scale xy plane in its own figure.
    plane = cuts["plane"]
    values_yx = plane_values_yx(plane)
    variable_label = plane["variable"].label or plane["variable"].name
    zmin, zmax = (None, None) if color_range is None else color_range

    fig = go.Figure(
        go.Heatmap(
            x=plane["x"],
            y=plane["y"],
            z=values_yx,
            colorscale=colorscale,
            zmin=zmin,
            zmax=zmax,
            zsmooth=False,
            connectgaps=False,
            hoverongaps=False,
            colorbar={"title": variable_label},
        )
    )
    fig.update_layout(
        title=title or f"{cuts['path'].name}: {cuts['variable']}",
        xaxis_title=plane["x_label"],
        yaxis_title=plane["y_label"],
        template="plotly_white",
        height=560,
        margin={"l": 60, "r": 50, "t": 75, "b": 55},
        meta={
            "requested_plane_nm": plane.get("requested_slice_coordinate"),
            "actual_plane_nm": plane["slice_coordinate"],
            "linear_scale": True,
        },
    )
    fig.update_yaxes(scaleanchor="x", scaleratio=1)
    return fig


def plot_vtr_line(cuts, *, title=None):
    # Plot one raw, linear-scale line cut in its own figure.
    line = cuts["line"]
    variable_label = line["variable"].label or line["variable"].name
    fixed_text = ", ".join(
        f"{name}={value:.3f} nm"
        for name, value in line["chosen_coords"].items()
    )
    fig = go.Figure(
        go.Scatter(
            x=line["axis"],
            y=np.asarray(line["values"], dtype=float),
            mode="lines",
            name=cuts["variable"],
        )
    )
    fig.update_layout(
        title=title or f"{cuts['path'].name}: {cuts['variable']} ({fixed_text})",
        xaxis_title=line["axis_label"],
        yaxis_title=variable_label,
        template="plotly_white",
        height=480,
        showlegend=False,
        margin={"l": 70, "r": 35, "t": 75, "b": 55},
        meta={"chosen_fixed_coordinates_nm": line["chosen_coords"], "linear_scale": True},
    )
    return fig

### 7.1 Requested VTR outputs and exact QW cut coordinates

All planes below are `xy` cuts at exactly `z = -3.750 nm`. All values are
shown on linear scales without smoothing or interpolation.

In [41]:
CUT_PLANE_AXIS = "z"
CUT_PLANE_VALUE_NM = -3.75
CUT_LINE_AXIS = "x"
CUT_LINE_FIXED_COORDS_NM = {"y": 100.0, "z": -3.75}

VTR_OUTPUTS = [
    {
        "label": "Hole density (density_hole.vtr)",
        "path": ANALYSIS_BIAS_DIR / "density_hole.vtr",
        "variable": "Hole_density",
        "shared_scale_group": "density",
    },
    {
        "label": "Electrostatic potential (potential.vtr)",
        "path": ANALYSIS_BIAS_DIR / "potential.vtr",
        "variable": "Potential",
        "shared_scale_group": None,
    },
    {
        "label": "Quantum HH density (density_c-Ge_QW_HH.vtr)",
        "path": ANALYSIS_RUN_DIR / "density_c-Ge_QW_HH.vtr",
        "variable": "Density",
        "shared_scale_group": "density",
    },
    *[
        {
            "label": f"Shifted HH probability, state {state}",
            "path": ANALYSIS_QUANTUM_DIR
            / f"probability_shift_c-Ge_QW_HH_{state:04d}.vtr",
            "variable": f"Psi^2_{state}",
            "shared_scale_group": None,
        }
        for state in (1, 2, 3)
    ],
]

vtr_manifest = pd.DataFrame(
    {
        "label": item["label"],
        "path": str(item["path"]),
        "exists": item["path"].exists(),
        "variable": item["variable"],
        "available_variables": (
            nnt.list_variables(item["path"])
            if item["path"].exists()
            else []
        ),
    }
    for item in VTR_OUTPUTS
)
display(vtr_manifest)
assert vtr_manifest["exists"].all()

,label,path,exists,variable,available_variables
0,Hole density (density_hole.vtr),/Users/robertjovanov/code/qpu-design-automatio...,True,Hole_density,[Hole_density]
1,Electrostatic potential (potential.vtr),/Users/robertjovanov/code/qpu-design-automatio...,True,Potential,[Potential]
2,Quantum HH density (density_c-Ge_QW_HH.vtr),/Users/robertjovanov/code/qpu-design-automatio...,True,Density,[Density]
3,"Shifted HH probability, state 1",/Users/robertjovanov/code/qpu-design-automatio...,True,Psi^2_1,"[E_1, Psi^2_1]"
4,"Shifted HH probability, state 2",/Users/robertjovanov/code/qpu-design-automatio...,True,Psi^2_2,"[E_2, Psi^2_2]"
5,"Shifted HH probability, state 3",/Users/robertjovanov/code/qpu-design-automatio...,True,Psi^2_3,"[E_3, Psi^2_3]"


### 7.2 Separate raw plane and line plots

The two density planes use one shared linear color range, making their spatial
patterns directly comparable. The `probability_shift` fields include their
state-energy offsets and are plotted exactly as written by nextnano.

In [42]:
# Extract all data first so shared color limits can be determined before plotting.
vtr_cut_results = {
    output["label"]: extract_vtr_cuts(
        output["path"],
        variable=output["variable"],
        plane_axis=CUT_PLANE_AXIS,
        plane_value_nm=CUT_PLANE_VALUE_NM,
        line_axis=CUT_LINE_AXIS,
        line_fixed_coords_nm=CUT_LINE_FIXED_COORDS_NM,
    )
    for output in VTR_OUTPUTS
}

density_outputs = [
    output for output in VTR_OUTPUTS
    if output["shared_scale_group"] == "density"
]
SHARED_DENSITY_COLOR_RANGE = (
    0.0,
    max(
        float(np.nanmax(plane_values_yx(vtr_cut_results[output["label"]]["plane"])))
        for output in density_outputs
    ),
)
print("Shared raw density color range:", SHARED_DENSITY_COLOR_RANGE)

cut_consistency_rows = []
for output in VTR_OUTPUTS:
    cuts = vtr_cut_results[output["label"]]
    plane = cuts["plane"]
    line = cuts["line"]
    plane_yx = plane_values_yx(plane)
    line_y_coordinate = line["chosen_coords"].get("y")
    plane_y_index = int(np.argmin(np.abs(np.asarray(plane["y"]) - line_y_coordinate)))
    line_matches_plane = np.array_equal(
        plane_yx[plane_y_index, :],
        np.asarray(line["values"]),
    )
    cut_consistency_rows.append(
        {
            "output": output["label"],
            "actual_plane_z_nm": plane["slice_coordinate"],
            "line_y_nm": line_y_coordinate,
            "line_z_nm": line["chosen_coords"].get("z"),
            "line_matches_plane_row_exactly": line_matches_plane,
            "finite_plane_values": bool(np.isfinite(plane_yx).all()),
        }
    )

    color_range = (
        SHARED_DENSITY_COLOR_RANGE
        if output["shared_scale_group"] == "density"
        else None
    )
    display(
        plot_vtr_plane(
            cuts,
            title=f"{output['label']}: xy at z = -3.750 nm",
            color_range=color_range,
        )
    )
    display(
        plot_vtr_line(
            cuts,
            title=f"{output['label']}: x line at y = 100 nm, z = -3.750 nm",
        )
    )

cut_consistency = pd.DataFrame(cut_consistency_rows)
display(cut_consistency)
assert cut_consistency["line_matches_plane_row_exactly"].all()
assert cut_consistency["finite_plane_values"].all()

Shared raw density color range: (0.0, 3.48849293711e+17)


,output,actual_plane_z_nm,line_y_nm,line_z_nm,line_matches_plane_row_exactly,finite_plane_values
0,Hole density (density_hole.vtr),-3.75,100.0,-3.75,True,True
1,Electrostatic potential (potential.vtr),-3.75,100.0,-3.75,True,True
2,Quantum HH density (density_c-Ge_QW_HH.vtr),-3.75,100.0,-3.75,True,True
3,"Shifted HH probability, state 1",-3.75,100.0,-3.75,True,True
4,"Shifted HH probability, state 2",-3.75,100.0,-3.75,True,True
5,"Shifted HH probability, state 3",-3.75,100.0,-3.75,True,True


### 7.3 Classical/quantum density consistency diagnostic

This checks the raw planes rather than their rendered colors. The two files are
different physical quantities and need not have identical amplitudes, but they
must use the same grid and should have consistent spatial structure.

In [43]:
classical_label = "Hole density (density_hole.vtr)"
quantum_label = "Quantum HH density (density_c-Ge_QW_HH.vtr)"
classical_plane = vtr_cut_results[classical_label]["plane"]
quantum_plane = vtr_cut_results[quantum_label]["plane"]
classical_values = plane_values_yx(classical_plane)
quantum_values = plane_values_yx(quantum_plane)

assert np.array_equal(classical_plane["x"], quantum_plane["x"])
assert np.array_equal(classical_plane["y"], quantum_plane["y"])
assert classical_values.shape == quantum_values.shape

significant = (classical_values > 1e12) & (quantum_values > 1e12)
spatial_correlation = float(
    np.corrcoef(classical_values[significant], quantum_values[significant])[0, 1]
)

classical_max_yx = np.unravel_index(np.argmax(classical_values), classical_values.shape)
quantum_max_yx = np.unravel_index(np.argmax(quantum_values), quantum_values.shape)

def xy_at_yx_index(plane, index_yx):
    iy, ix = index_yx
    return float(plane["x"][ix]), float(plane["y"][iy])


density_consistency_summary = pd.Series(
    {
        "same_x_grid": np.array_equal(classical_plane["x"], quantum_plane["x"]),
        "same_y_grid": np.array_equal(classical_plane["y"], quantum_plane["y"]),
        "same_plane_shape": classical_values.shape == quantum_values.shape,
        "all_classical_pixels_finite": bool(np.isfinite(classical_values).all()),
        "all_quantum_pixels_finite": bool(np.isfinite(quantum_values).all()),
        "classical_max_cm^-3": float(np.max(classical_values)),
        "quantum_HH_max_cm^-3": float(np.max(quantum_values)),
        "classical_max_xy_nm": xy_at_yx_index(classical_plane, classical_max_yx),
        "quantum_max_xy_nm": xy_at_yx_index(quantum_plane, quantum_max_yx),
        "significant_pixel_spatial_correlation": spatial_correlation,
    },
    name="value",
)
display(density_consistency_summary.to_frame())

,value
same_x_grid,True
same_y_grid,True
same_plane_shape,True
all_classical_pixels_finite,True
all_quantum_pixels_finite,True
classical_max_cm^-3,157780374195000000.0
quantum_HH_max_cm^-3,348849293711000000.0
classical_max_xy_nm,"(0.0, 0.0)"
quantum_max_xy_nm,"(0.0, 0.0)"
significant_pixel_spatial_correlation,0.940172


## 8. HH occupation and energy spectrum

In [44]:
QUANTUM_REGION = "c-Ge_QW"
QUANTUM_BAND = "HH"
QUANTUM_KPOINT = "k00000"

occupation = nnt.read_quantum_occupation(
    ANALYSIS_RUN_DIR,
    region=QUANTUM_REGION,
    band=QUANTUM_BAND,
    bias=BIAS_INDEX,
).rename(columns=lambda column: "state" if column == "no." else column)

energy_spectrum = nnt.read_quantum_energy_spectrum(
    ANALYSIS_RUN_DIR,
    region=QUANTUM_REGION,
    band=QUANTUM_BAND,
    kpoint=QUANTUM_KPOINT,
    bias=BIAS_INDEX,
).rename(columns=lambda column: "state" if column == "no." else column)

occupation_column = next(column for column in occupation if column != "state")
energy_column = next(column for column in energy_spectrum if column != "state")
quantum_state_table = energy_spectrum.merge(occupation, on="state", how="outer")

display(quantum_state_table)

occupied_mask = occupation[occupation_column] > 1e-6
quantum_summary = pd.Series(
    {
        "number_of_reported_states": len(quantum_state_table),
        "total_HH_occupation_holes": float(occupation[occupation_column].sum()),
        "states_with_occupation_above_1e-6": int(occupied_mask.sum()),
        "highest_state_with_occupation_above_1e-6": int(
            occupation.loc[occupied_mask, "state"].max()
        ),
        "energy_min_eV": float(energy_spectrum[energy_column].min()),
        "energy_max_eV": float(energy_spectrum[energy_column].max()),
        "states_with_positive_energy": int((energy_spectrum[energy_column] > 0).sum()),
    },
    name="value",
)
display(quantum_summary.to_frame())

,state,Energy[eV],Occupation[holes]
0,1,0.001150,1.998607e+00
1,2,0.001005,1.998565e+00
2,3,0.000851,1.998706e+00
3,4,0.000768,1.998574e+00
4,5,0.000737,1.998666e+00
...,...,...,...
95,96,-0.003086,1.207273e-304
96,97,-0.003100,1.265672e-304
97,98,-0.003142,1.247520e-304
98,99,-0.003316,1.188287e-304


,value
number_of_reported_states,100.000000
total_HH_occupation_holes,37.112771
states_with_occupation_above_1e-6,19.000000
highest_state_with_occupation_above_1e-6,19.000000
energy_min_eV,-0.003322
energy_max_eV,0.001150
states_with_positive_energy,19.000000


In [45]:
occupation_figure = nnt.plot_quantum_occupation(
    ANALYSIS_RUN_DIR,
    region=QUANTUM_REGION,
    band=QUANTUM_BAND,
    bias=BIAS_INDEX,
    interactive=True,
)

energy_spectrum_figure = nnt.plot_quantum_energy_spectrum(
    ANALYSIS_RUN_DIR,
    region=QUANTUM_REGION,
    band=QUANTUM_BAND,
    kpoint=QUANTUM_KPOINT,
    bias=BIAS_INDEX,
    interactive=True,
)